In [1]:
import json
import hashlib, json
from bitcoinutils.setup import setup
from bitcoinutils.keys import PrivateKey, PublicKey, P2trAddress
from bitcoinutils.transactions import Transaction, TxInput, TxOutput, TxWitnessInput
from bitcoinutils.script import Script
from coincurve import PrivateKey as ccPrivateKey , PublicKeyXOnly 
from pathlib import Path
from bitcoinutils.utils import tapleaf_tagged_hash, tapbranch_tagged_hash

setup("regtest")


'regtest'

In [27]:
path_proof = 'files/batch_size/8/doc_5_proof.json'
deliverable = json.loads(Path(path_proof).read_text())


In [28]:
from bitcoinrpc.authproxy import AuthServiceProxy
from pprint import pprint


RPC_USER = "ghost"
RPC_PASS = "ghost"
RPC_PORT = 18443
RPC_HOST = "127.0.0.1"

rpc = AuthServiceProxy(f"http://{RPC_USER}:{RPC_PASS}@{RPC_HOST}:{RPC_PORT}")

print(rpc.getblockchaininfo()["chain"])
pprint(rpc.getblockchaininfo())

regtest
{'bestblockhash': '10ee19d9e2d226bba6204ed121bc8df589bb3c4d9d5ab7b890213436fc9dce50',
 'blocks': 103,
 'chain': 'regtest',
 'chainwork': '00000000000000000000000000000000000000000000000000000000000000d0',
 'difficulty': Decimal('4.656542373906925E-10'),
 'headers': 103,
 'initialblockdownload': False,
 'mediantime': 1776435866,
 'pruned': False,
 'size_on_disk': 31459,
 'time': 1776475335,
 'verificationprogress': 1,
 'warnings': ''}


In [29]:
rpc = AuthServiceProxy(f"http://{RPC_USER}:{RPC_PASS}@{RPC_HOST}:{RPC_PORT}")

rawtx = deliverable['idtx']
decoded = rpc.getrawtransaction(rawtx, 1)

for output in decoded['vout']:
    # Buscamos dentro de scriptPubKey
    if 'address' in output['scriptPubKey']:
        if output['scriptPubKey']['address'] == deliverable['address']:
            utxo_base =  output['n']

print(rawtx)        
utxo_spend = rpc.gettxout(rawtx,int(utxo_base))
print(utxo_spend)


5f7d21efce3c8c7e5ddd228414b304f8193d5be84a423480c7df1a04730f3bb9
None


In [30]:
def find_child_by_same_address(rpc, address: str, parent_txid: str):
    res = rpc.scantxoutset("start", [f"addr({address})"])
    for u in res.get("unspents", []):
        if u["txid"] != parent_txid:
            return {
                "child_txid": u["txid"],
                "child_vout": u["vout"],
                "amount": u["amount"],
                "raw": u,
            }
    return None

rpc = AuthServiceProxy(f"http://{RPC_USER}:{RPC_PASS}@{RPC_HOST}:{RPC_PORT}")

address_check = deliverable['address']
idtx_check = deliverable['idtx']
result = find_child_by_same_address(
    rpc,
    address_check,
    idtx_check
)

result['child_txid']


'4dd9abce6621b1ba252525f51415c7778ca197b2c5d39d5c7a40e26b19eac96e'

In [31]:

rpc = AuthServiceProxy(f"http://{RPC_USER}:{RPC_PASS}@{RPC_HOST}:{RPC_PORT}")


tx = rpc.getrawtransaction(result['child_txid'], 1)

revocation_hex = tx['vout'][1]['scriptPubKey']['hex'][4:]
revocation_bitmap = bin(int(revocation_hex, 16))[2:].zfill(len(revocation_hex) * 4)

if revocation_bitmap[-deliverable['index']]:
    print('Revoqued')
else: 
    print('Not Revoqued')



Revoqued


In [32]:
deliverable['index']


5

In [33]:
revocation_hex = tx['vout'][1]['scriptPubKey']['hex'][4:]
revocation_bitmap


'11110000'

In [37]:
revocation_bitmap[-deliverable['index']]

'1'

In [ ]:
if revocation_bitmap[-deliverable['index']]=='1':
    print('Revoqued')
else: 
    print('Not Revoqued')

Not Revoqued
